# MediCare Hospital — Patient Analytics & Readmission Prediction




In [1]:
# libraries we need
%pip install pandas numpy scikit-learn sqlalchemy
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sqlalchemy import create_engine


Note: you may need to restart the kernel to use updated packages.


## 1. Extract


In [2]:
patients = pd.read_csv("patients.csv")
admissions = pd.read_csv("admissions.csv", parse_dates=["admission_date", "discharge_date"])
vitals = pd.read_csv("vitals.csv", parse_dates=["recorded_time"])
treatments = pd.read_csv("treatments.csv", parse_dates=["treatment_date"])
doctors = pd.read_csv("doctors.csv")

for name, df in [("patients", patients), ("admissions", admissions), ("vitals", vitals),
                  ("treatments", treatments), ("doctors", doctors)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")


patients: 1690 rows, 8 columns
admissions: 3324 rows, 7 columns
vitals: 33148 rows, 7 columns
treatments: 8329 rows, 6 columns
doctors: 40 rows, 5 columns


## 2. Profile (before touching anything)

Same discipline as the other case studies. This dataset in particular has a couple of real
issues worth finding before they quietly bias a readmission model.


In [3]:
# missing values per table
for name, df in [("patients", patients), ("admissions", admissions), ("vitals", vitals),
                  ("treatments", treatments), ("doctors", doctors)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f"{name}: no missing values")
    else:
        print(f"{name}: missing values -")
        print(missing)


patients: missing values -
chronic_conditions    240
dtype: int64
admissions: no missing values
vitals: no missing values
treatments: no missing values
doctors: no missing values


In [4]:
# a missing discharge_date most likely means the patient is still admitted, not that
# the value is genuinely unknown - worth confirming the count before assuming either way
still_admitted = admissions["discharge_date"].isnull().sum()
print(f"admissions with no discharge_date (likely still admitted): {still_admitted} of {len(admissions)}")


admissions with no discharge_date (likely still admitted): 0 of 3324


In [5]:
# duplicate key checks
print("duplicate patient_id:", patients["patient_id"].duplicated().sum())
print("duplicate admission_id:", admissions["admission_id"].duplicated().sum())
print("duplicate doctor_id:", doctors["doctor_id"].duplicated().sum())


duplicate patient_id: 0
duplicate admission_id: 0
duplicate doctor_id: 0


In [6]:
# referential integrity - every admission should point at a real patient and doctor
missing_patients = set(admissions["patient_id"]) - set(patients["patient_id"])
missing_doctors = set(admissions["doctor_id"]) - set(doctors["doctor_id"])
missing_admissions_v = set(vitals["admission_id"]) - set(admissions["admission_id"])
missing_admissions_t = set(treatments["admission_id"]) - set(admissions["admission_id"])

print("admissions with no matching patient:", len(missing_patients))
print("admissions with no matching doctor:", len(missing_doctors))
print("vitals with no matching admission:", len(missing_admissions_v))
print("treatments with no matching admission:", len(missing_admissions_t))


admissions with no matching patient: 0
admissions with no matching doctor: 0
vitals with no matching admission: 0
treatments with no matching admission: 0


In [7]:
# range / sanity checks on the vitals - flag anything outside a physically plausible range
print("heart_rate outside 30-220 bpm:", ((vitals["heart_rate"] < 30) | (vitals["heart_rate"] > 220)).sum())
print("oxygen_level outside 50-100%:", ((vitals["oxygen_level"] < 50) | (vitals["oxygen_level"] > 100)).sum())
print("temperature outside 90-110F:", ((vitals["temperature"] < 90) | (vitals["temperature"] > 110)).sum())


heart_rate outside 30-220 bpm: 0
oxygen_level outside 50-100%: 0
temperature outside 90-110F: 0


In [8]:
# a real data quality issue in this dataset: vitals and treatments recorded on dates
# that fall outside the admission's own admission_date/discharge_date window. For a
# patient still admitted (no discharge_date yet), only the lower bound can be checked.
v = vitals.merge(admissions[["admission_id", "admission_date", "discharge_date"]], on="admission_id")
v_before_admission = (v["recorded_time"].dt.normalize() < v["admission_date"]).sum()
v_after_discharge = ((v["discharge_date"].notnull()) & (v["recorded_time"].dt.normalize() > v["discharge_date"])).sum()
print(f"vitals recorded before the patient was admitted: {v_before_admission} of {len(v)}")
print(f"vitals recorded after the patient was discharged: {v_after_discharge} of {len(v)}")

t = treatments.merge(admissions[["admission_id", "admission_date", "discharge_date"]], on="admission_id")
t_before_admission = (t["treatment_date"] < t["admission_date"]).sum()
t_after_discharge = ((t["discharge_date"].notnull()) & (t["treatment_date"] > t["discharge_date"])).sum()
print(f"treatments dated before the patient was admitted: {t_before_admission} of {len(t)}")
print(f"treatments dated after the patient was discharged: {t_after_discharge} of {len(t)}")


vitals recorded before the patient was admitted: 3437 of 33148
vitals recorded after the patient was discharged: 29064 of 33148
treatments dated before the patient was admitted: 4243 of 8329
treatments dated after the patient was discharged: 3925 of 8329


These out-of-window rows are a real limitation of this sample data (treatment and
vitals timestamps weren't generated to line up with each admission's actual date range).
They're kept rather than dropped, since dropping a large share of vitals/treatments would
lose more signal than it's worth for a case study — but this is exactly the kind of thing
you'd flag to whoever owns the source EHR system in a real project, not quietly work around.


In [9]:
# gender in the patients file doesn't always match the first name it's attached to
# (e.g. "Stephen" recorded as Female) - there's no way to know which is actually correct
# from this data alone, so it's used as given and just noted here rather than "fixed"
# by guessing from the name, which would just be introducing a different kind of error
print(patients["gender"].value_counts())


gender
Female    886
Male      804
Name: count, dtype: int64


## 3. Transform & Feature Engineering

This is the core of the notebook: building a real 30-day readmission label from the
admissions history, then a feature table to predict it with.


In [10]:
# 30-day readmission is the standard clinical definition - did the same patient have
# another admission within 30 days of this one's discharge?
admissions = admissions.sort_values(["patient_id", "admission_date"]).reset_index(drop=True)

def flag_readmission(group):
    group = group.sort_values("admission_date").reset_index(drop=True)
    readmitted = []
    for i in range(len(group)):
        this_discharge = group.loc[i, "discharge_date"]
        if pd.isnull(this_discharge):
            readmitted.append(np.nan)   # still admitted - outcome not known yet, exclude later
            continue
        later_admissions = group.loc[i+1:, "admission_date"]
        within_30_days = ((later_admissions - this_discharge).dt.days <= 30).any()
        readmitted.append(int(within_30_days))
    # return just the key + flag, rather than the whole group - merging this back by
    # admission_id afterward sidesteps a pandas groupby/apply quirk where the grouping
    # column (patient_id) can get silently dropped from the group passed into this function
    return pd.DataFrame({"admission_id": group["admission_id"], "readmitted_30d": readmitted})

flags = (
    admissions.groupby("patient_id", group_keys=False)[["admission_id", "admission_date", "discharge_date"]]
    .apply(flag_readmission)
)
admissions = admissions.merge(flags, on="admission_id")

print(admissions["readmitted_30d"].value_counts(dropna=False))


readmitted_30d
0    2893
1     431
Name: count, dtype: int64


In [11]:
# patient age at the time of admission
patients["dob"] = pd.to_datetime(patients["dob"])
admissions = admissions.merge(patients[["patient_id", "dob", "gender", "chronic_conditions"]], on="patient_id")
admissions["age_at_admission"] = ((admissions["admission_date"] - admissions["dob"]).dt.days // 365).astype(int)

# number of chronic conditions ("None" counts as zero)
admissions["chronic_condition_count"] = admissions["chronic_conditions"].apply(
    lambda x: 0 if pd.isnull(x) or str(x).strip().lower() == "none" else len(str(x).split(","))
)

# length of stay in days (NaN for patients still admitted)
admissions["length_of_stay"] = (admissions["discharge_date"] - admissions["admission_date"]).dt.days

# how many admissions this patient has had before this one
admissions["past_admissions_count"] = admissions.groupby("patient_id").cumcount()


In [12]:
# summarize vitals per admission: average and worst readings, plus a simple
# abnormal-vitals flag count (heart rate out of 60-100, oxygen below 92%, fever above 100.4F)
vitals["is_abnormal"] = (
    (vitals["heart_rate"] < 60) | (vitals["heart_rate"] > 100) |
    (vitals["oxygen_level"] < 92) |
    (vitals["temperature"] > 100.4)
).astype(int)

vitals_summary = vitals.groupby("admission_id").agg(
    avg_heart_rate=("heart_rate", "mean"),
    avg_oxygen_level=("oxygen_level", "mean"),
    max_temperature=("temperature", "max"),
    abnormal_vitals_count=("is_abnormal", "sum"),
    vitals_recorded_count=("vital_id", "count"),
).reset_index()

admissions = admissions.merge(vitals_summary, on="admission_id", how="left")


In [13]:
# number of treatments given during the admission
treatment_counts = treatments.groupby("admission_id").size().reset_index(name="treatment_count")
admissions = admissions.merge(treatment_counts, on="admission_id", how="left")
admissions["treatment_count"] = admissions["treatment_count"].fillna(0)

admissions.head()


,admission_id,patient_id,admission_date,discharge_date,diagnosis,doctor_id,room_no,readmitted_30d,dob,gender,...,age_at_admission,chronic_condition_count,length_of_stay,past_admissions_count,avg_heart_rate,avg_oxygen_level,max_temperature,abnormal_vitals_count,vitals_recorded_count,treatment_count
0,2001,1001,2026-02-07,2026-02-13,Diabetes Complication,317,R418,1,1976-12-15,Female,...,49,1,6,0,91.000000,93.888889,100.48,8,9,4
1,2002,1001,2026-03-07,2026-03-17,Heart Failure,320,R396,0,1976-12-15,Female,...,49,1,10,1,90.916667,92.666667,100.85,8,12,1
2,2003,1002,2025-05-16,2025-05-21,Asthma Attack,339,R411,0,1964-06-11,Female,...,60,1,5,0,83.866667,94.200000,100.41,9,15,4
3,2005,1003,2025-08-11,2025-08-21,Pneumonia,306,R368,0,1939-09-03,Female,...,85,0,10,0,90.400000,95.000000,100.86,6,10,2
4,2004,1003,2025-12-08,2025-12-10,Respiratory Infection,325,R120,0,1939-09-03,Female,...,86,0,2,1,83.777778,92.222222,100.60,7,9,2


## 4. Predictive Model — Readmission Risk

Only admissions with a known outcome (i.e. the patient has actually been discharged) can be
used to train or score the model — a currently-admitted patient's 30-day outcome isn't known
yet, so including them would be guessing at the label, not predicting from features.


In [14]:
model_data = admissions.dropna(subset=["readmitted_30d"]).copy()
model_data["readmitted_30d"] = model_data["readmitted_30d"].astype(int)

feature_cols = ["age_at_admission", "chronic_condition_count", "length_of_stay",
                 "past_admissions_count", "avg_heart_rate", "avg_oxygen_level",
                 "max_temperature", "abnormal_vitals_count", "treatment_count"]

# a handful of admissions may have no vitals or treatments recorded - fill with the
# column median rather than dropping them
for col in feature_cols:
    model_data[col] = model_data[col].fillna(model_data[col].median())

X = model_data[feature_cols]
y = model_data["readmitted_30d"]

print("training rows:", len(X))
print("readmission rate in this data:", round(y.mean() * 100, 1), "%")


training rows: 3324
readmission rate in this data: 13.0 %


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)   # tree models don't need scaled features


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [16]:
# compare the two models on the held-out test set
for name, model, X_eval in [("Logistic Regression", log_reg, X_test_scaled), ("Random Forest", rf, X_test)]:
    preds = model.predict(X_eval)
    print(f"\n{name}")
    print("  accuracy:", round(accuracy_score(y_test, preds), 3))
    print("  precision:", round(precision_score(y_test, preds, zero_division=0), 3))
    print("  recall:", round(recall_score(y_test, preds, zero_division=0), 3))
    print("  confusion matrix:\n", confusion_matrix(y_test, preds))



Logistic Regression
  accuracy: 0.87
  precision: 0.0
  recall: 0.0
  confusion matrix:
 [[723   0]
 [108   0]]

Random Forest
  accuracy: 0.87
  precision: 0.0
  recall: 0.0
  confusion matrix:
 [[723   0]
 [108   0]]


## 5. Score every discharged admission and build the Readmission_Risk table

This is the table the brief describes but that doesn't exist as a source file — it's
generated here, matching the schema given: `risk_id`, `admission_id`, `prediction_date`,
`risk_score`, `risk_level`.


In [20]:
# use whichever model had the better recall in Section 4 - swap this if Random Forest wins instead
chosen_model = log_reg
X_all_scaled = scaler.transform(model_data[feature_cols])
model_data["risk_score"] = chosen_model.predict_proba(X_all_scaled)[:, 1].round(2)

def risk_level(score):
    if score >= 0.150:
        return "High"
    elif score >= 0.120:
        return "Medium"
    else:
        return "Low"

model_data["risk_level"] = model_data["risk_score"].apply(risk_level)

readmission_risk = model_data[["admission_id", "risk_score", "risk_level"]].copy()
readmission_risk.insert(0, "risk_id", range(9001, 9001 + len(readmission_risk)))
readmission_risk["prediction_date"] = datetime.today().date()

readmission_risk.head()


,risk_id,admission_id,risk_score,risk_level,prediction_date
0,9001,2001,0.18,High,2026-10-10
1,9002,2002,0.11,Low,2026-10-10
2,9003,2003,0.17,High,2026-10-10
3,9004,2005,0.24,High,2026-10-10
4,9005,2004,0.07,Low,2026-10-10


## 6. Load into MySQL

Note: `patients`, `admissions`, `vitals`, `treatments`, and `doctors` are loaded via
`LOAD DATA LOCAL INFILE` in the SQL script, not from here — that keeps the SQL script
self-contained and matches how the other case studies were set up. This notebook only
writes the one new table it created: `readmission_risk`.


In [18]:
import urllib.parse
user = "root"
password = urllib.parse.quote_plus("Dell@2026")
host = "localhost"
database = "healthcare_analytics_db"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

readmission_risk.to_sql("readmission_risk", engine, if_exists="replace", index=False)
print(f"loaded readmission_risk: {len(readmission_risk)} rows")


loaded readmission_risk: 3324 rows


## 7. Export clean CSVs for Power BI

If Power BI doesn't have a MySQL connector available, every table it needs can be loaded
directly from these CSVs instead (Get Data → Text/CSV, or Get Data → Folder to grab all of
them at once) — no database connection required either way.


In [19]:
patients.to_csv("clean_patients.csv", index=False)
admissions.to_csv("clean_admissions.csv", index=False)
vitals.to_csv("clean_vitals.csv", index=False)
treatments.to_csv("clean_treatments.csv", index=False)
doctors.to_csv("clean_doctors.csv", index=False)
readmission_risk.to_csv("clean_readmission_risk.csv", index=False)

print("Saved: clean_patients.csv, clean_admissions.csv, clean_vitals.csv, "
      "clean_treatments.csv, clean_doctors.csv, clean_readmission_risk.csv")


Saved: clean_patients.csv, clean_admissions.csv, clean_vitals.csv, clean_treatments.csv, clean_doctors.csv, clean_readmission_risk.csv
